# Model Promotion: Idempotent Model Copy Between Catalogs

This notebook promotes a RAG model from one environment to another:
- **dev → qa**: Promotes latest dev model to QA catalog
- **qa → prod**: Promotes validated QA model to Production catalog

## Idempotency
- ✅ Checks if model already exists in target
- ✅ Compares source and target versions
- ✅ Skips copy if versions match
- ✅ Safe to re-run multiple times

In [0]:
# Databricks notebook source
import mlflow
from mlflow import MlflowClient

In [0]:
dbutils.widgets.text("src_catalog", "rag_agentic")
dbutils.widgets.text("tgt_catalog", "workday_sales_qa")
dbutils.widgets.text("schema_name", "workday_demos")
dbutils.widgets.text("model_name", "workday_sales_rag")

src_catalog  = dbutils.widgets.get("src_catalog")
tgt_catalog  = dbutils.widgets.get("tgt_catalog")
schema_name = dbutils.widgets.get("schema_name")
model_name   = dbutils.widgets.get("model_name")

src_model_name = f"{src_catalog}.{schema_name}.{model_name}"
tgt_model_name = f"{tgt_catalog}.{schema_name}.{model_name}"

In [0]:
def get_latest_model_version(model_name):
    """Get the latest version number of a registered model"""
    try:
        versions = client.search_model_versions(f"name='{model_name}'")
        if not versions:
            return None
        return max([int(v.version) for v in versions])
    except Exception as e:
        return None

def get_model_version_details(model_name, version):
    """Get details of a specific model version"""
    try:
        return client.get_model_version(model_name, version)
    except Exception:
        return None

def model_versions_match(src_model, src_version, dst_model, dst_version):
    """Check if source and destination model versions match"""
    src_details = get_model_version_details(src_model, src_version)
    tgt_details = get_model_version_details(dst_model, dst_version)
    
    if not src_details or not tgt_details:
        return False
    
    # Compare run IDs (unique identifier for model artifact)
    return src_details.run_id == tgt_details.run_id

print("✅ Helper functions loaded")

In [0]:
client = MlflowClient()

# Get latest version from source
print(f"\n{'='*70}")
print("STEP 1: Check Source Model Exists or not")
print(f"{'='*70}\n")

src_latest_version = get_latest_model_version(src_model_name)
if not src_latest_version:
    print(f"❌ ERROR: No model found in source: {src_model_name}")
    dbutils.notebook.exit({"status": "failed", "error": "Source model not found"})

print(f"Source model found: {src_model_name}, latest version: {src_latest_version}")

# Get source model details
src_model_details = get_model_version_details(src_model_name, src_latest_version)
print(f"   Run ID: {src_model_details.run_id}, status: {src_model_details.status}")

In [0]:
# Get latest version from target
print(f"\n{'='*70}")
print("STEP 2: Check Target Model (Idempotency Check)")
print(f"{'='*70}\n")

tgt_latest_version = get_latest_model_version(tgt_model_name)
if tgt_latest_version:
    print(f"Target model exists: {tgt_model_name}, Latest version: {tgt_latest_version}")    
    # Check if versions match
    if model_versions_match(src_model_name, src_latest_version, tgt_model_name, tgt_latest_version):
        print(f"Source v{src_latest_version} == Target v{tgt_latest_version}, and Same run_id: {src_model_details.run_id}. Therefore, skipping promotion")     
        dbutils.notebook.exit({
                                    "status": "success",
                                    "action": "skipped",
                                    "reason": "Model already promoted",
                                    "source_version": src_latest_version,
                                    "target_version": tgt_latest_version
                                })
    else:
        print(f"Target model exists but versions don't match, create new version in target")
else:
    print(f"Target model does not exist yet, create new model and first version")

In [0]:
print(f"\n{'='*70}")
print("STEP 3: Copy Model to Target")
print(f"{'='*70}\n")

try:
    # Copy model version to target catalog
    print(f"📦 Copying model version...")
    print(f"   From: {src_model_name} (v{src_latest_version})")
    print(f"   To: {tgt_model_name}")
    
    copied_version = client.copy_model_version(
                                                src_model_uri=f"models:/{src_model_name}/{src_latest_version}",
                                                dst_name=tgt_model_name
                                                )
    
    new_version = copied_version.version
    
    print(f"Model copied successfully! \n New version in target: v{new_version} \n Run ID: {copied_version.run_id}")    
except Exception as e:
    print(f"\n ERROR: Failed to copy model, {str(e)}")
    dbutils.notebook.exit({"status": "failed", "error": str(e)})

In [0]:
print(f"\n{'='*70}")
print("STEP 4: Update Model Alias")
print(f"{'='*70}\n")

# Set alias based on destination environment
if tgt_catalog == "workday_sales_qa":
    alias = "Candidate"
elif tgt_catalog == "workday_sales_prod":
    alias = "Champion"
else:
    alias = None

if alias:
    try:
        # Set or update alias
        client.set_registered_model_alias(
                                            name=tgt_model_name,
                                            alias=alias,
                                            version=new_version
                                        )
        print(f"Alias '{alias}' set to version {new_version}")
    except Exception as e:
        print(f"Warning: Could not set alias: {str(e)}")
        # Don't fail the job if alias update fails
else:
    print(f"No alias update needed for the model exists in {tgt_catalog} catalog")

In [0]:
print(f"\n{'='*70}")
print(f"✅ MODEL PROMOTION COMPLETE!")
print(f"{'='*70}")

print(f"📊 Summary: \n Source: {src_model_name} v{src_latest_version} \n Target: {tgt_model_name} v{new_version}")
if alias:
    print(f"Alias: @{alias} → v{new_version}")

# Exit with success status
dbutils.notebook.exit({
                        "status": "success",
                        "action": "promoted",
                        "source_model": src_model_name,
                        "source_version": src_latest_version,
                        "target_model": tgt_model_name,
                        "target_version": new_version,
                        "alias": alias
                    })